<a href="https://colab.research.google.com/github/Nantakomar/TraigeLevel_Prediction/blob/main/Healthcare_Final_VIVA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CW2 Task 2 - Healthcare Triage ML Model

Dataset: synthetic_medical_triage.csv

Location: Google Drive / CW2_AI_Healthcare / Datasets

Tool: Google Colab with TensorFlow/Keras

Purpose: Build a simple ML component to classify patient triage priority.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/CW2_AI_Healthcare/Datasets/synthetic_medical_triage.csv"

df = pd.read_csv(file_path)

df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns)

print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
df['triage_level'].value_counts().sort_index()

In [ ]:
df['triage_level'].value_counts(normalize=True).sort_index() * 100

In [ ]:
df.info()
df.describe()

In [ ]:
df['arrival_mode'].value_counts()

In [ ]:
df_encoded = pd.get_dummies(df, columns=['arrival_mode'], dtype=int)

df_encoded.head()

In [ ]:
df_encoded.info()

In [ ]:
selected_features = [
    'age',
    'heart_rate',
    'systolic_blood_pressure',
    'oxygen_saturation',
    'body_temperature',
    'pain_level'
]

X = df_encoded[selected_features]
y = df_encoded['triage_level']

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nSelected input features:")
print(X.columns.tolist())

print("\nTarget column:")
print(y.name)

In [ ]:
# ============================================================
# REPRODUCIBILITY SETTINGS
# ============================================================

import os
import random
import numpy as np
import tensorflow as tf

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Make TensorFlow operations deterministic where possible
tf.config.experimental.enable_op_determinism()

print("Random seed fixed at:", SEED)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split the dataset into training and testing data
# stratify=y keeps the same triage level distribution in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale the selected input features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train shape:", X_train_scaled.shape)
print("X_test shape:", X_test_scaled.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).sort_index() * 100)

print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True).sort_index() * 100)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Clear any previous model from memory
tf.keras.backend.clear_session()

# Training parameters
learning_rate = 0.001
epochs = 50
batch_size = 32

# Build the neural network
model = keras.Sequential([
    layers.Input(shape=(6,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(4, activation="softmax")
])

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=learning_rate
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Display architecture and parameters
model.summary()

print("Learning rate:", learning_rate)
print("Epochs:", epochs)
print("Batch size:", batch_size)

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=epochs,
    batch_size=batch_size,
    shuffle=True,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

# Plot training and validation accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Plot training and validation loss
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import matplotlib.pyplot as plt

# Predict the test set
y_pred_probs = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1, 2, 3])
disp.plot()
plt.title("Confusion Matrix for Triage Level Classification")
plt.show()

In [ ]:
# Example patient input for prediction
example_patient = pd.DataFrame([{
    'age': 70,
    'heart_rate': 125,
    'systolic_blood_pressure': 88,
    'oxygen_saturation': 89,
    'body_temperature': 39.2,
    'pain_level': 8
}])

# Scale the example patient using the same scaler
example_patient_scaled = scaler.transform(example_patient)

# Predict triage level
prediction_probabilities = model.predict(example_patient_scaled)
predicted_class = np.argmax(prediction_probabilities, axis=1)[0]

print("Example patient data:")
print(example_patient)

print("\nPrediction probabilities:")
print(prediction_probabilities)

print("\nPredicted triage level:", predicted_class)

In [ ]:
import os
import joblib

# Folder in Google Drive
save_folder = "/content/drive/MyDrive/CW2_AI_Healthcare"

# Create the folder if it does not already exist
os.makedirs(save_folder, exist_ok=True)

# File locations
model_path = os.path.join(
    save_folder,
    "final_triage_model.keras"
)

scaler_path = os.path.join(
    save_folder,
    "final_scaler.pkl"
)

# Save the trained neural network
model.save(model_path)

# Save the fitted StandardScaler
joblib.dump(scaler, scaler_path)

print("Model saved successfully:")
print(model_path)

print("\nScaler saved successfully:")
print(scaler_path)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import tensorflow as tf
import joblib

model_path = (
    "/content/drive/MyDrive/"
    "CW2_AI_Healthcare/"
    "final_triage_model.keras"
)

scaler_path = (
    "/content/drive/MyDrive/"
    "CW2_AI_Healthcare/"
    "final_scaler.pkl"
)

loaded_model = tf.keras.models.load_model(model_path)
loaded_scaler = joblib.load(scaler_path)

print("Saved model and scaler loaded successfully.")

In [ ]:
import pandas as pd
import numpy as np

example_patient = pd.DataFrame([{
    "age": 70,
    "heart_rate": 125,
    "systolic_blood_pressure": 88,
    "oxygen_saturation": 89,
    "body_temperature": 39.2,
    "pain_level": 8
}])

example_scaled = loaded_scaler.transform(example_patient)

probabilities = loaded_model.predict(
    example_scaled,
    verbose=0
)[0]

predicted_level = int(np.argmax(probabilities))

for level, probability in enumerate(probabilities):
    print(f"Triage Level {level}: {probability:.2%}")

print("Predicted triage level:", predicted_level)

In [ ]:
loaded_model.summary()